> **Public release note.** Notebook outputs have been removed because the underlying Malaysian Motor claims data are confidential. Execution counts have also been cleared to provide clean public versions of the notebooks. Local user-specific paths and individual claim identifiers have been removed. The repository documents the data-processing, modelling and validation workflow used in the dissertation, but the numerical results cannot be reproduced end-to-end without the confidential input data.


# 05.9B — Historical Pure IBNR and Integrated Framework

**Purpose:** complete the historical 2021 Q4 valuation exercise by adding:

1. a leakage-free macro pure IBNR estimate to the reported-claim
   Random Forest and XGBoost projections;
2. a separate early-maturity macro treatment for the current accident year,
   which was below DEV_QTR_4 at the valuation date;
3. like-for-like total-portfolio comparisons with Incurred Chain Ladder.

The pure IBNR method uses a **rolling three-accident-year,
same-maturity frequency–severity estimate**. For each target accident year,
the method uses the preceding three accident years observed at the same
maturity and follows them for the same four-quarter run-off horizon.

Example for target AY2020 at valuation date 2021 Q4:

- AY2019 observed from 2020 Q4 to 2021 Q4;
- AY2018 observed from 2019 Q4 to 2020 Q4;
- AY2017 observed from 2018 Q4 to 2019 Q4.

Every comparator outcome date is no later than 2021 Q4, so the estimate uses
only information that would have been available at the historical valuation
date.

this cell makes sure the previous RF/XGBoost/Chain Ladder back-test outputs are available and prepares the notebook to add the separate pure IBNR component needed for a like-for-like Integrated Framework comparison.

In [ ]:
from pathlib import Path
import gc
import json
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_FOLDER = Path(
    "/path/to/BI_large_claims_project"
)

HISTORICAL_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_9A_historical_valuation_diagonal"
)

OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_9B_historical_pure_ibnr_integrated_framework"
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

ROLLING_COMPARATOR_YEARS = 3
STREAM_BATCH_SIZE = 250_000

required_historical_files = {
    "portfolio": HISTORICAL_FOLDER / "historical_diagonal_portfolio_summary.csv",
    "accident_year": HISTORICAL_FOLDER / "historical_diagonal_comparison_by_accident_year.csv",
    "scope": HISTORICAL_FOLDER / "historical_diagonal_outcome_scope_reconciliation.csv",
    "deployment": HISTORICAL_FOLDER / "historical_diagonal_deployment_audit.csv",
    "cl_factors": HISTORICAL_FOLDER / "historical_incurred_chain_ladder_factors.csv",
}

for name, path in required_historical_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} input: {path}")

print("Historical input folder:", HISTORICAL_FOLDER)
print("Output folder:", OUTPUT_FOLDER)

## Load the completed historical diagonal

The dates are read from the existing portfolio summary rather than entered
independently. This prevents the Integrated Framework from using a
different valuation cut-off from the RF, XGBoost and Chain Ladder results.

This section loads the saved results from the previous historical back-test. It uses these results to find the exact valuation dates T0 and T1, the four-quarter run-off horizon, and the accident years included in the analysis. The section also checks that there is one consistent pair of valuation dates and gives a warning if the horizon is not four quarters as expected.





this cell  makes sure the pure IBNR calculation is aligned with exactly the same historical period and accident-year scope used in the previous RF, XGBoost and Chain Ladder comparison.

In [ ]:
portfolio_existing = pd.read_csv(required_historical_files["portfolio"])
ay_existing = pd.read_csv(required_historical_files["accident_year"])
scope_existing = pd.read_csv(required_historical_files["scope"])
deployment_existing = pd.read_csv(required_historical_files["deployment"])
cl_factors = pd.read_csv(required_historical_files["cl_factors"])

def quarter_label_to_index(label):
    year_text, quarter_text = str(label).strip().split()
    year = int(year_text)
    quarter = int(quarter_text.replace("Q", ""))
    return 4 * year + quarter

def quarter_index_to_label(index_value):
    index_value = int(index_value)
    year = (index_value - 1) // 4
    quarter = index_value - 4 * year
    return f"{year} Q{quarter}"

t0_labels = portfolio_existing["T0"].dropna().unique()
t1_labels = portfolio_existing["T1"].dropna().unique()

if len(t0_labels) != 1 or len(t1_labels) != 1:
    raise ValueError("Historical portfolio summary does not contain unique T0/T1 dates.")

T0_LABEL = str(t0_labels[0])
T1_LABEL = str(t1_labels[0])
T0_INDEX = quarter_label_to_index(T0_LABEL)
T1_INDEX = quarter_label_to_index(T1_LABEL)
HORIZON_QTRS = T1_INDEX - T0_INDEX

if HORIZON_QTRS != 4:
    warnings.warn(
        f"This notebook was designed for a four-quarter run-off horizon, "
        f"but the input horizon is {HORIZON_QTRS} quarters."
    )

TARGET_AYS = sorted(ay_existing["ACC_YEAR"].astype(int).unique())
CURRENT_AY = (T0_INDEX - 1) // 4

print("T0:", T0_LABEL, T0_INDEX)
print("T1:", T1_LABEL, T1_INDEX)
print("Main diagonal AY range:", min(TARGET_AYS), "to", max(TARGET_AYS))
print("Current accident year requiring early treatment:", CURRENT_AY)

## Required historical observation dates

A three-year rolling estimate requires year-end positions from 2018 Q4
through 2022 Q4. The exact dates are generated from \(T_0\), so the method
also remains reusable if the historical valuation date is changed later.

In [ ]:
historical_snapshot_indices = [
    T0_INDEX - 4 * lag
    for lag in range(1, ROLLING_COMPARATOR_YEARS + 1)
]
historical_outcome_indices = [
    snapshot_index + HORIZON_QTRS
    for snapshot_index in historical_snapshot_indices
]

REQUIRED_POSITION_INDICES = sorted(
    set(
        [T0_INDEX, T1_INDEX]
        + historical_snapshot_indices
        + historical_outcome_indices
    )
)

required_dates_df = pd.DataFrame({
    "valuation_index": REQUIRED_POSITION_INDICES,
    "valuation_label": [
        quarter_index_to_label(index_value)
        for index_value in REQUIRED_POSITION_INDICES
    ],
})

display(required_dates_df)

if max(historical_outcome_indices) > T0_INDEX:
    raise ValueError(
        "Comparator outcomes extend beyond T0 and would introduce leakage."
    )

## Stream the master parquet

Only the fields needed to identify reporting dates and BI Excess positions
are read. The full 12-million-row dataset is processed in batches.

A claim is treated as reported from the first valuation quarter in which
`CLAIMS_CNT > 0`. Pure IBNR at a historical snapshot therefore means:

- first report date after the snapshot;
- first report date no later than the comparison outcome;
- positive incurred BI Excess at the outcome date.

In [ ]:
parquet_file = pq.ParquetFile(MASTER_PATH)
available_columns = set(parquet_file.schema_arrow.names)

REQUIRED_MASTER_COLUMNS = [
    "SOURCE_FILE",
    "CLAIMS_KEY",
    "ACC_YEAR",
    "ACC_QTR",
    "DEV_QTR",
    "CLAIMS_CNT",
    "CUM_INC_LARGE",
]

missing_columns = sorted(
    set(REQUIRED_MASTER_COLUMNS).difference(available_columns)
)
if missing_columns:
    raise ValueError(
        f"Master parquet is missing required fields: {missing_columns}"
    )

first_report_batches = []
position_batches = {
    valuation_index: []
    for valuation_index in REQUIRED_POSITION_INDICES
}

rows_scanned = 0

for batch_number, record_batch in enumerate(
    parquet_file.iter_batches(
        batch_size=STREAM_BATCH_SIZE,
        columns=REQUIRED_MASTER_COLUMNS,
    ),
    start=1,
):
    chunk = record_batch.to_pandas()
    rows_scanned += len(chunk)

    for col in ["ACC_YEAR", "ACC_QTR", "DEV_QTR", "CLAIMS_CNT", "CUM_INC_LARGE"]:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk["CLAIMS_CNT"] = chunk["CLAIMS_CNT"].fillna(0.0)
    chunk["CUM_INC_LARGE"] = chunk["CUM_INC_LARGE"].fillna(0.0)

    chunk["ACCIDENT_QTR_INDEX"] = (
        4 * chunk["ACC_YEAR"].astype(int)
        + chunk["ACC_QTR"].astype(int)
    )
    chunk["VALUATION_QTR_INDEX"] = (
        chunk["ACCIDENT_QTR_INDEX"]
        + chunk["DEV_QTR"].astype(int)
    )

    report_rows = chunk.loc[
        chunk["CLAIMS_CNT"] > 0,
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
            "VALUATION_QTR_INDEX",
        ],
    ]

    if not report_rows.empty:
        first_report_batch = (
            report_rows
            .groupby(
                ["SOURCE_FILE", "CLAIMS_KEY", "ACC_YEAR", "ACC_QTR"],
                as_index=False,
            )
            .agg(first_report_index=("VALUATION_QTR_INDEX", "min"))
        )
        first_report_batches.append(first_report_batch)

    required_rows = chunk.loc[
        chunk["VALUATION_QTR_INDEX"].isin(REQUIRED_POSITION_INDICES),
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
            "VALUATION_QTR_INDEX",
            "CUM_INC_LARGE",
            "CLAIMS_CNT",
        ],
    ]

    if not required_rows.empty:
        for valuation_index, date_rows in required_rows.groupby(
            "VALUATION_QTR_INDEX"
        ):
            position_batches[int(valuation_index)].append(
                date_rows.copy()
            )

    del chunk, report_rows, required_rows, record_batch

    if batch_number % 10 == 0:
        gc.collect()
        print(f"Scanned {rows_scanned:,} master rows...")

if not first_report_batches:
    raise ValueError("No reported claims were identified in the master data.")

first_report = pd.concat(first_report_batches, ignore_index=True)
first_report = (
    first_report
    .groupby(
        ["SOURCE_FILE", "CLAIMS_KEY", "ACC_YEAR", "ACC_QTR"],
        as_index=False,
    )
    .agg(first_report_index=("first_report_index", "min"))
)

del first_report_batches
gc.collect()

positions = {}

for valuation_index in REQUIRED_POSITION_INDICES:
    if not position_batches[valuation_index]:
        raise ValueError(
            f"No rows were found for valuation index {valuation_index} "
            f"({quarter_index_to_label(valuation_index)})."
        )

    position_df = pd.concat(
        position_batches[valuation_index],
        ignore_index=True,
    )

    position_df = (
        position_df
        .sort_values(
            [
                "SOURCE_FILE",
                "CLAIMS_KEY",
                "VALUATION_QTR_INDEX",
            ]
        )
        .drop_duplicates(
            ["SOURCE_FILE", "CLAIMS_KEY"],
            keep="last",
        )
    )

    position_df = position_df.merge(
        first_report[
            [
                "SOURCE_FILE",
                "CLAIMS_KEY",
                "first_report_index",
            ]
        ],
        on=["SOURCE_FILE", "CLAIMS_KEY"],
        how="left",
        validate="one_to_one",
    )

    position_df["reported_by_position"] = (
        position_df["first_report_index"].notna()
        & (position_df["first_report_index"] <= valuation_index)
    )

    positions[valuation_index] = position_df

    print(
        quarter_index_to_label(valuation_index),
        "position rows:",
        f"{len(position_df):,}",
        "| reported:",
        f"{int(position_df['reported_by_position'].sum()):,}",
        "| BIXS:",
        f"{position_df['CUM_INC_LARGE'].sum():,.2f}",
    )

del position_batches
gc.collect()

## Cohort measurement function

For any accident year and historical snapshot/outcome pair, the function
separates:

- reported-claim development;
- pure IBNR BI Excess;
- total one-year BI Excess development.

In [ ]:
KEY_COLUMNS = ["SOURCE_FILE", "CLAIMS_KEY"]

def cohort_metrics(accident_year, snapshot_index, outcome_index):
    snapshot = positions[snapshot_index].loc[
        positions[snapshot_index]["ACC_YEAR"].eq(accident_year)
    ].copy()

    outcome = positions[outcome_index].loc[
        positions[outcome_index]["ACC_YEAR"].eq(accident_year)
    ].copy()

    cohort_reports = first_report.loc[
        first_report["ACC_YEAR"].eq(accident_year)
    ].copy()

    reported_keys = cohort_reports.loc[
        cohort_reports["first_report_index"] <= snapshot_index,
        KEY_COLUMNS,
    ].drop_duplicates()

    reported_claim_count = len(reported_keys)

    snapshot_reported = snapshot.merge(
        reported_keys.assign(reported_at_snapshot=True),
        on=KEY_COLUMNS,
        how="inner",
    )

    outcome_with_report = outcome.merge(
        cohort_reports[
            KEY_COLUMNS + ["first_report_index"]
        ],
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
        suffixes=("", "_report"),
    )

    reported_outcome = outcome_with_report.loc[
        outcome_with_report["first_report_index"] <= snapshot_index
    ]

    pure_ibnr_outcome = outcome_with_report.loc[
        (outcome_with_report["first_report_index"] > snapshot_index)
        & (outcome_with_report["first_report_index"] <= outcome_index)
        & (outcome_with_report["CUM_INC_LARGE"] > 0)
    ]

    late_reported_all = outcome_with_report.loc[
        (outcome_with_report["first_report_index"] > snapshot_index)
        & (outcome_with_report["first_report_index"] <= outcome_index)
    ]

    snapshot_reported_bixs = float(
        snapshot_reported["CUM_INC_LARGE"].sum()
    )
    reported_outcome_bixs = float(
        reported_outcome["CUM_INC_LARGE"].sum()
    )
    full_outcome_bixs = float(
        outcome_with_report["CUM_INC_LARGE"].sum()
    )
    pure_ibnr_bixs = float(
        pure_ibnr_outcome["CUM_INC_LARGE"].sum()
    )

    return {
        "accident_year": int(accident_year),
        "snapshot_index": int(snapshot_index),
        "snapshot_label": quarter_index_to_label(snapshot_index),
        "outcome_index": int(outcome_index),
        "outcome_label": quarter_index_to_label(outcome_index),
        "reported_claim_count_at_snapshot": int(reported_claim_count),
        "snapshot_reported_bixs": snapshot_reported_bixs,
        "reported_claim_bixs_at_outcome": reported_outcome_bixs,
        "reported_claim_future_development": (
            reported_outcome_bixs - snapshot_reported_bixs
        ),
        "pure_ibnr_positive_bixs_claim_count": int(
            len(pure_ibnr_outcome)
        ),
        "pure_ibnr_bixs_amount": pure_ibnr_bixs,
        "late_reported_claim_count_all_outcomes": int(
            len(late_reported_all)
        ),
        "full_portfolio_bixs_at_outcome": full_outcome_bixs,
        "total_future_development": (
            full_outcome_bixs - snapshot_reported_bixs
        ),
    }

## Build same-maturity rolling comparator experience

For target accident year \(a\), comparator \(a-k\) is observed at
\(T_0-4k\) and followed to \(T_0-4k+4\). This holds development maturity
approximately constant while ensuring the complete comparator outcome was
known at \(T_0\).

In [ ]:
all_target_ays = sorted(set(TARGET_AYS + [CURRENT_AY]))
minimum_data_ay = int(first_report["ACC_YEAR"].min())

comparator_rows = []

for target_ay in all_target_ays:
    for lag in range(1, ROLLING_COMPARATOR_YEARS + 1):
        comparator_ay = target_ay - lag

        if comparator_ay < minimum_data_ay:
            continue

        snapshot_index = T0_INDEX - 4 * lag
        outcome_index = snapshot_index + HORIZON_QTRS

        if outcome_index > T0_INDEX:
            raise ValueError(
                "Comparator outcome occurs after T0 and introduces leakage."
            )

        metrics = cohort_metrics(
            comparator_ay,
            snapshot_index,
            outcome_index,
        )

        comparator_rows.append({
            "target_accident_year": target_ay,
            "comparator_lag_years": lag,
            "comparator_accident_year": comparator_ay,
            **metrics,
        })

comparator_detail = pd.DataFrame(comparator_rows)

comparator_detail.to_csv(
    OUTPUT_FOLDER / "historical_pure_ibnr_comparator_detail.csv",
    index=False,
)

display(comparator_detail.tail(12))

## Estimate pure IBNR by accident year

The pooled historical frequency and severity are:

$$
\widehat{f}_{\mathrm{AY}}
=
\frac{
\sum \text{late positive-BIXS claims}
}{
\sum \text{reported claims at comparator snapshots}
}
$$

$$
\widehat{s}_{\mathrm{AY}}
=
\frac{
\sum \text{late-reported BI Excess amount}
}{
\sum \text{late positive-BIXS claims}
}
$$

The target accident-year allowance is:

$$
\widehat{U}^{\mathrm{Pure\ IBNR}}_{\mathrm{AY}}
=
N^{\mathrm{Reported}}_{\mathrm{AY},T_0}
\widehat{f}_{\mathrm{AY}}
\widehat{s}_{\mathrm{AY}}
$$

Pooling the prior three cohorts stabilises the sparse frequency and severity
estimates while preserving the maturity structure.

In [ ]:
pure_ibnr_estimate_rows = []

for target_ay in all_target_ays:
    target_metrics = cohort_metrics(
        target_ay,
        T0_INDEX,
        T1_INDEX,
    )

    comparators = comparator_detail.loc[
        comparator_detail["target_accident_year"].eq(target_ay)
    ].copy()

    comparator_exposure = float(
        comparators["reported_claim_count_at_snapshot"].sum()
    )
    comparator_pure_count = float(
        comparators["pure_ibnr_positive_bixs_claim_count"].sum()
    )
    comparator_pure_amount = float(
        comparators["pure_ibnr_bixs_amount"].sum()
    )

    if comparator_exposure > 0:
        frequency = comparator_pure_count / comparator_exposure
        amount_per_reported_claim = (
            comparator_pure_amount / comparator_exposure
        )
    else:
        frequency = 0.0
        amount_per_reported_claim = 0.0

    severity = (
        comparator_pure_amount / comparator_pure_count
        if comparator_pure_count > 0
        else 0.0
    )

    estimated_count = (
        target_metrics["reported_claim_count_at_snapshot"]
        * frequency
    )
    estimated_amount = (
        target_metrics["reported_claim_count_at_snapshot"]
        * amount_per_reported_claim
    )

    actual_amount = target_metrics["pure_ibnr_bixs_amount"]

    pure_ibnr_estimate_rows.append({
        "ACC_YEAR": target_ay,
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "comparator_accident_years": ",".join(
            str(int(value))
            for value in comparators[
                "comparator_accident_year"
            ].tolist()
        ),
        "comparator_year_count": len(comparators),
        "target_reported_claim_count_at_T0": (
            target_metrics["reported_claim_count_at_snapshot"]
        ),
        "comparator_reported_claim_exposure": comparator_exposure,
        "comparator_pure_ibnr_positive_claim_count": (
            comparator_pure_count
        ),
        "comparator_pure_ibnr_amount": comparator_pure_amount,
        "estimated_pure_ibnr_frequency": frequency,
        "estimated_pure_ibnr_severity": severity,
        "estimated_pure_ibnr_amount_per_reported_claim": (
            amount_per_reported_claim
        ),
        "estimated_pure_ibnr_claim_count": estimated_count,
        "estimated_pure_ibnr_bixs_amount": estimated_amount,
        "observed_pure_ibnr_positive_claim_count_T1": (
            target_metrics["pure_ibnr_positive_bixs_claim_count"]
        ),
        "observed_pure_ibnr_bixs_amount_T1": actual_amount,
        "pure_ibnr_amount_error": estimated_amount - actual_amount,
        "pure_ibnr_absolute_error": abs(estimated_amount - actual_amount),
    })

pure_ibnr_by_ay = pd.DataFrame(pure_ibnr_estimate_rows)

pure_ibnr_by_ay.to_csv(
    OUTPUT_FOLDER / "historical_pure_ibnr_estimate_by_accident_year.csv",
    index=False,
)

display(pure_ibnr_by_ay)

## Reconcile the observed pure IBNR amount

The newly derived monetary amount should agree with the historical
scope-reconciliation file. The positive-BIXS claim count is intentionally
more precise than the earlier `observed_pure_ibnr_claim_count`, which counted
all late-reported claims irrespective of whether they had positive BI Excess.

In [ ]:
pure_main = pure_ibnr_by_ay.loc[
    pure_ibnr_by_ay["ACC_YEAR"].isin(TARGET_AYS)
].copy()

pure_reconciliation = pure_main.merge(
    scope_existing[
        [
            "ACC_YEAR",
            "observed_pure_ibnr_bixs_T1",
            "observed_pure_ibnr_claim_count",
        ]
    ],
    on="ACC_YEAR",
    how="left",
    validate="one_to_one",
)

pure_reconciliation["amount_difference_new_minus_existing"] = (
    pure_reconciliation["observed_pure_ibnr_bixs_amount_T1"]
    - pure_reconciliation["observed_pure_ibnr_bixs_T1"]
)

max_amount_difference = float(
    pure_reconciliation[
        "amount_difference_new_minus_existing"
    ].abs().max()
)

print(
    "Maximum pure IBNR amount reconciliation difference:",
    f"{max_amount_difference:,.2f}",
)

if max_amount_difference > 1.0:
    warnings.warn(
        "The newly derived pure IBNR amount differs from the earlier "
        "scope reconciliation. Review the reporting-date definition."
    )

pure_reconciliation.to_csv(
    OUTPUT_FOLDER / "historical_pure_ibnr_reconciliation_audit.csv",
    index=False,
)

display(
    pure_reconciliation[
        [
            "ACC_YEAR",
            "estimated_pure_ibnr_bixs_amount",
            "observed_pure_ibnr_bixs_amount_T1",
            "observed_pure_ibnr_positive_claim_count_T1",
            "observed_pure_ibnr_claim_count",
            "amount_difference_new_minus_existing",
        ]
    ]
)

## Add pure IBNR to Random Forest and XGBoost

The Integrated Framework for the main diagonal accident years is:

$$
\widehat{U}^{M,\mathrm{Integrated}}_{\mathrm{AY}}
=
\widehat{U}^{M,\mathrm{Reported}}_{\mathrm{AY}}
+
\widehat{U}^{\mathrm{Pure\ IBNR}}_{\mathrm{AY}},
$$

where \(M\) is Random Forest or XGBoost.

This produces a like-for-like total-portfolio estimate for comparison with
Incurred Chain Ladder.

In [ ]:
integrated_ay = ay_existing.merge(
    pure_main[
        [
            "ACC_YEAR",
            "estimated_pure_ibnr_bixs_amount",
            "observed_pure_ibnr_bixs_amount_T1",
        ]
    ],
    on="ACC_YEAR",
    how="left",
    validate="one_to_one",
)

integrated_ay["estimated_pure_ibnr_bixs_amount"] = (
    integrated_ay["estimated_pure_ibnr_bixs_amount"].fillna(0.0)
)

integrated_ay["rf_integrated_projected_latest_bixs"] = (
    integrated_ay["rf_projected_latest_bixs"]
    + integrated_ay["estimated_pure_ibnr_bixs_amount"]
)
integrated_ay["xgb_integrated_projected_latest_bixs"] = (
    integrated_ay["xgb_projected_latest_bixs"]
    + integrated_ay["estimated_pure_ibnr_bixs_amount"]
)

for method in ["rf", "xgb"]:
    projected_col = f"{method}_integrated_projected_latest_bixs"
    error_col = f"{method}_integrated_total_error"

    integrated_ay[error_col] = (
        integrated_ay[projected_col]
        - integrated_ay["full_portfolio_actual_latest_bixs_T1"]
    )
    integrated_ay[f"{method}_integrated_total_absolute_error"] = (
        integrated_ay[error_col].abs()
    )
    integrated_ay[f"{method}_integrated_total_percentage_error"] = np.where(
        integrated_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
        integrated_ay[error_col]
        / integrated_ay["full_portfolio_actual_latest_bixs_T1"],
        np.nan,
    )

integrated_ay.to_csv(
    OUTPUT_FOLDER / "historical_integrated_framework_by_accident_year.csv",
    index=False,
)

display(
    integrated_ay[
        [
            "ACC_YEAR",
            "rf_projected_latest_bixs",
            "estimated_pure_ibnr_bixs_amount",
            "rf_integrated_projected_latest_bixs",
            "xgb_integrated_projected_latest_bixs",
            "cl_projected_latest_bixs_T1",
            "full_portfolio_actual_latest_bixs_T1",
            "rf_integrated_total_error",
            "xgb_integrated_total_error",
            "cl_total_error",
        ]
    ]
)

In [ ]:
def summarise_method(
    method,
    scope,
    projected_column,
    observed_column,
    absolute_error_column,
):
    observed = float(integrated_ay[observed_column].sum())
    projected = float(integrated_ay[projected_column].sum())
    error = projected - observed
    ay_wape = (
        float(integrated_ay[absolute_error_column].sum()) / abs(observed)
        if observed != 0
        else np.nan
    )

    return {
        "method": method,
        "scope": scope,
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "accident_year_min": int(integrated_ay["ACC_YEAR"].min()),
        "accident_year_max": int(integrated_ay["ACC_YEAR"].max()),
        "observed_latest_bixs_T1": observed,
        "projected_latest_bixs_T1": projected,
        "difference_projected_minus_observed": error,
        "bias_ratio": projected / observed if observed != 0 else np.nan,
        "accident_year_wape": ay_wape,
    }

integrated_portfolio_summary = pd.DataFrame([
    summarise_method(
        "Random Forest Integrated Framework",
        "Total portfolio: reported claims plus estimated pure IBNR",
        "rf_integrated_projected_latest_bixs",
        "full_portfolio_actual_latest_bixs_T1",
        "rf_integrated_total_absolute_error",
    ),
    summarise_method(
        "XGBoost Integrated Framework",
        "Total portfolio: reported claims plus estimated pure IBNR",
        "xgb_integrated_projected_latest_bixs",
        "full_portfolio_actual_latest_bixs_T1",
        "xgb_integrated_total_absolute_error",
    ),
    summarise_method(
        "Incurred Chain Ladder",
        "Total portfolio",
        "cl_projected_latest_bixs_T1",
        "full_portfolio_actual_latest_bixs_T1",
        "cl_total_absolute_error",
    ),
])

integrated_portfolio_summary.to_csv(
    OUTPUT_FOLDER / "historical_integrated_framework_portfolio_summary.csv",
    index=False,
)

display(integrated_portfolio_summary)

## Early-maturity macro treatment for the current accident year
The current accident year was below DEV_QTR_4 at $T_0$, so it does not
receive a Random Forest or XGBoost prediction.

A separate macro estimate uses the same prior three accident years at the
same maturity. For each comparator accident year, total future development is:

$$
\text{Full BI Excess at outcome}
-
\text{reported BI Excess at snapshot}
$$

This amount includes both development on reported claims and pure IBNR. The
pooled future-development amount per reported claim is then applied to the
current accident year.

in this cell for the youngest accident year, where claim-level ML is not yet available, it uses a simple portfolio-level estimate based on how similar immature accident years developed over the same horizon.

In [ ]:
early_comparators = comparator_detail.loc[
    comparator_detail["target_accident_year"].eq(CURRENT_AY)
].copy()

early_target_metrics = cohort_metrics(
    CURRENT_AY,
    T0_INDEX,
    T1_INDEX,
)

early_comparator_exposure = float(
    early_comparators["reported_claim_count_at_snapshot"].sum()
)
early_comparator_total_future = float(
    early_comparators["total_future_development"].sum()
)
early_comparator_reported_future = float(
    early_comparators["reported_claim_future_development"].sum()
)
early_comparator_pure_ibnr = float(
    early_comparators["pure_ibnr_bixs_amount"].sum()
)

early_amount_per_reported_claim = (
    early_comparator_total_future / early_comparator_exposure
    if early_comparator_exposure > 0
    else 0.0
)

early_estimated_future = (
    early_target_metrics["reported_claim_count_at_snapshot"]
    * early_amount_per_reported_claim
)
early_projected_latest = (
    early_target_metrics["snapshot_reported_bixs"]
    + early_estimated_future
)
early_observed_latest = (
    early_target_metrics["full_portfolio_bixs_at_outcome"]
)

early_macro_result = pd.DataFrame([{
    "ACC_YEAR": CURRENT_AY,
    "T0": T0_LABEL,
    "T1": T1_LABEL,
    "comparator_accident_years": ",".join(
        str(int(value))
        for value in early_comparators[
            "comparator_accident_year"
        ].tolist()
    ),
    "target_reported_claim_count_at_T0": (
        early_target_metrics["reported_claim_count_at_snapshot"]
    ),
    "target_reported_bixs_at_T0": (
        early_target_metrics["snapshot_reported_bixs"]
    ),
    "comparator_reported_claim_exposure": early_comparator_exposure,
    "comparator_total_future_development": (
        early_comparator_total_future
    ),
    "comparator_reported_claim_future_development": (
        early_comparator_reported_future
    ),
    "comparator_pure_ibnr_bixs": early_comparator_pure_ibnr,
    "estimated_total_future_per_reported_claim": (
        early_amount_per_reported_claim
    ),
    "estimated_total_future_development": early_estimated_future,
    "macro_projected_latest_bixs": early_projected_latest,
    "observed_latest_bixs_T1": early_observed_latest,
    "difference_projected_minus_observed": (
        early_projected_latest - early_observed_latest
    ),
    "bias_ratio": (
        early_projected_latest / early_observed_latest
        if early_observed_latest != 0
        else np.nan
    ),
    "observed_reported_claim_future_development": (
        early_target_metrics["reported_claim_future_development"]
    ),
    "observed_pure_ibnr_bixs": (
        early_target_metrics["pure_ibnr_bixs_amount"]
    ),
    "observed_total_future_development": (
        early_target_metrics["total_future_development"]
    ),
}])

early_macro_result.to_csv(
    OUTPUT_FOLDER / "historical_early_maturity_macro_current_ay.csv",
    index=False,
)

display(early_macro_result.T)

## Extend Incurred Chain Ladder to the current accident year

The earlier Chain Ladder output excluded the current accident year only
because the ML framework had no under-DEV_QTR_4 component. The early macro
estimate now permits a complete AY2010–current-year comparison.

The existing historical Chain Ladder factors are applied to each current
accident quarter from its maturity at \(T_0\) to its maturity at \(T_1\).

this cell gives the Chain Ladder benchmark for the youngest accident year, using the same \(T_0\) to \(T_1\) horizon as the early-maturity macro estimate, so the two approaches can be compared on a consistent basis.

In [ ]:
factor_lookup = (
    cl_factors
    .set_index("development_quarter")["factor"]
    .to_dict()
)
usable_lookup = (
    cl_factors
    .set_index("development_quarter")["factor_usable"]
    .to_dict()
)

t0_current_position = positions[T0_INDEX].loc[
    positions[T0_INDEX]["ACC_YEAR"].eq(CURRENT_AY)
].copy()

current_by_accident_quarter = (
    t0_current_position
    .groupby("ACC_QTR", as_index=False)
    .agg(incurred_bixs_T0=("CUM_INC_LARGE", "sum"))
)

current_cl_rows = []

for row in current_by_accident_quarter.itertuples(index=False):
    origin_index = 4 * CURRENT_AY + int(row.ACC_QTR)
    current_dev = T0_INDEX - origin_index
    target_dev = T1_INDEX - origin_index

    projected = float(row.incurred_bixs_T0)
    unusable_factor_count = 0

    for dev in range(current_dev, target_dev):
        factor = float(factor_lookup.get(dev, 1.0))
        if not bool(usable_lookup.get(dev, False)):
            unusable_factor_count += 1
        projected *= factor

    current_cl_rows.append({
        "ACC_YEAR": CURRENT_AY,
        "ACC_QTR": int(row.ACC_QTR),
        "development_at_T0": int(current_dev),
        "target_development_at_T1": int(target_dev),
        "incurred_bixs_T0": float(row.incurred_bixs_T0),
        "cl_projected_latest_bixs_T1": projected,
        "unusable_factor_count": unusable_factor_count,
    })

current_cl_projection = pd.DataFrame(current_cl_rows)

current_cl_projection.to_csv(
    OUTPUT_FOLDER / "historical_current_ay_cl_projection_by_accident_quarter.csv",
    index=False,
)

CURRENT_AY_CL_PROJECTED = float(
    current_cl_projection["cl_projected_latest_bixs_T1"].sum()
)

display(current_cl_projection)
print(
    "Current AY Chain Ladder projection:",
    f"{CURRENT_AY_CL_PROJECTED:,.2f}",
)

## Complete portfolio comparison including the current accident year

For AYs covered by the claim-level models, the complete Integrated Framework
is RF/XGBoost plus estimated pure IBNR.

For the current AY below DEV_QTR_4, the early-maturity macro projection is
used instead. This produces a complete portfolio estimate without pretending
that the RF model covers maturities for which it was not trained.

This section brings together the accident-year comparisons for all methods. It uses the main Integrated Framework results for older accident years and adds the early-maturity macro estimate for the current accident year. Then, it compares Random Forest, XGBoost, and Chain Ladder to the full observed BI Excess outcome.

In [ ]:
complete_ay = integrated_ay[
    [
        "ACC_YEAR",
        "full_portfolio_actual_latest_bixs_T1",
        "rf_integrated_projected_latest_bixs",
        "xgb_integrated_projected_latest_bixs",
        "cl_projected_latest_bixs_T1",
    ]
].copy()

current_complete_row = pd.DataFrame([{
    "ACC_YEAR": CURRENT_AY,
    "full_portfolio_actual_latest_bixs_T1": early_observed_latest,
    "rf_integrated_projected_latest_bixs": early_projected_latest,
    "xgb_integrated_projected_latest_bixs": early_projected_latest,
    "cl_projected_latest_bixs_T1": CURRENT_AY_CL_PROJECTED,
}])

complete_ay = pd.concat(
    [complete_ay, current_complete_row],
    ignore_index=True,
).sort_values("ACC_YEAR")

complete_ay["integrated_component"] = np.where(
    complete_ay["ACC_YEAR"].eq(CURRENT_AY),
    "Early-maturity macro",
    "Claim-level model plus pure IBNR",
)

for method in ["rf", "xgb"]:
    projected_col = f"{method}_integrated_projected_latest_bixs"
    complete_ay[f"{method}_complete_error"] = (
        complete_ay[projected_col]
        - complete_ay["full_portfolio_actual_latest_bixs_T1"]
    )
    complete_ay[f"{method}_complete_absolute_error"] = (
        complete_ay[f"{method}_complete_error"].abs()
    )
    complete_ay[f"{method}_complete_percentage_error"] = np.where(
        complete_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
        complete_ay[f"{method}_complete_error"]
        / complete_ay["full_portfolio_actual_latest_bixs_T1"],
        np.nan,
    )

complete_ay["cl_complete_error"] = (
    complete_ay["cl_projected_latest_bixs_T1"]
    - complete_ay["full_portfolio_actual_latest_bixs_T1"]
)
complete_ay["cl_complete_absolute_error"] = (
    complete_ay["cl_complete_error"].abs()
)
complete_ay["cl_complete_percentage_error"] = np.where(
    complete_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
    complete_ay["cl_complete_error"]
    / complete_ay["full_portfolio_actual_latest_bixs_T1"],
    np.nan,
)

complete_ay.to_csv(
    OUTPUT_FOLDER / "historical_complete_framework_by_accident_year.csv",
    index=False,
)

display(complete_ay)

This section gives the final portfolio-level comparison across all accident years. It adds up the observed and projected BI Excess amounts for the Random Forest Integrated Framework, XGBoost Integrated Framework, and Incurred Chain Ladder. It then calculates the overall error, bias ratio, and accident-year WAPE for each method.

In [ ]:
def complete_summary(method, projected_col, absolute_error_col):
    observed = float(
        complete_ay["full_portfolio_actual_latest_bixs_T1"].sum()
    )
    projected = float(complete_ay[projected_col].sum())

    return {
        "method": method,
        "scope": (
            f"Complete AY{int(complete_ay['ACC_YEAR'].min())}–"
            f"AY{int(complete_ay['ACC_YEAR'].max())} portfolio"
        ),
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "observed_latest_bixs_T1": observed,
        "projected_latest_bixs_T1": projected,
        "difference_projected_minus_observed": projected - observed,
        "bias_ratio": projected / observed if observed != 0 else np.nan,
        "accident_year_wape": (
            float(complete_ay[absolute_error_col].sum()) / abs(observed)
            if observed != 0
            else np.nan
        ),
    }

complete_portfolio_summary = pd.DataFrame([
    complete_summary(
        "Random Forest Integrated Framework",
        "rf_integrated_projected_latest_bixs",
        "rf_complete_absolute_error",
    ),
    complete_summary(
        "XGBoost Integrated Framework",
        "xgb_integrated_projected_latest_bixs",
        "xgb_complete_absolute_error",
    ),
    complete_summary(
        "Incurred Chain Ladder",
        "cl_projected_latest_bixs_T1",
        "cl_complete_absolute_error",
    ),
])

complete_portfolio_summary.to_csv(
    OUTPUT_FOLDER / "historical_complete_framework_portfolio_summary.csv",
    index=False,
)

display(complete_portfolio_summary)

This step completes the audit and reconciliation for the historical Integrated Framework notebook. It logs the valuation dates, comparator-year setting, accident-year range, estimated and observed pure IBNR, the reconciliation check, and the early-maturity current-year result. Finally, it saves the audit summary as a CSV file.

In [ ]:
audit_summary = pd.DataFrame([{
    "T0": T0_LABEL,
    "T1": T1_LABEL,
    "rolling_comparator_years": ROLLING_COMPARATOR_YEARS,
    "main_accident_year_min": min(TARGET_AYS),
    "main_accident_year_max": max(TARGET_AYS),
    "current_accident_year": CURRENT_AY,
    "estimated_pure_ibnr_main_total": float(
        pure_main["estimated_pure_ibnr_bixs_amount"].sum()
    ),
    "observed_pure_ibnr_main_total": float(
        pure_main["observed_pure_ibnr_bixs_amount_T1"].sum()
    ),
    "pure_ibnr_total_difference": float(
        pure_main["estimated_pure_ibnr_bixs_amount"].sum()
        - pure_main["observed_pure_ibnr_bixs_amount_T1"].sum()
    ),
    "maximum_amount_reconciliation_difference": max_amount_difference,
    "current_ay_macro_projection": early_projected_latest,
    "current_ay_observed_latest": early_observed_latest,
    "current_ay_macro_error": early_projected_latest - early_observed_latest,
}])

audit_summary.to_csv(
    OUTPUT_FOLDER / "historical_integrated_framework_audit_summary.csv",
    index=False,
)

display(audit_summary.T)

print("Saved all outputs to:", OUTPUT_FOLDER)